In [1]:
!pip install -q transformers soundfile accelerate scikit-learn

import os
import gc
import numpy as np
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from transformers import (
    WavLMModel,
    Wav2Vec2FeatureExtractor
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

VOX_PATH = (
    "/kaggle/input/datasets/arie07/"
    "audiodataset10percent/"
    "VoxCeleb/vox1_dev_wav"
)

BASELINE_CHECKPOINT = (
    "/kaggle/input/models/sd00ad/"
    "baselinemodelwavlm/pytorch/default/1/"
    "wavlm_baseline_local.pth"
)

PARTIAL_FT_CHECKPOINT = (
    "/kaggle/working/"
    "wavlm_partialft_local.pth"
)

TRAIN_SPEAKERS = sorted(
    [
        f"id{i}"
        for i in range(
            10041,
            11252
        )
    ]
)

spk2idx = {
    spk: idx
    for idx, spk in enumerate(
        TRAIN_SPEAKERS
    )
}

num_speakers = len(
    TRAIN_SPEAKERS
)

print(
    "Speakers:",
    num_speakers
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
import torch

CHECKPOINT_PATH = (
    "/kaggle/input/models/sd00ad/"
    "baselinemodelwavlm/pytorch/default/1/"
    "wavlm_baseline_local.pth"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu"
)

print(checkpoint.keys())
print("Step:", checkpoint["step"])

dict_keys(['model', 'optimizer', 'step'])
Step: 30000


In [3]:
class ASTP(nn.Module):

    def __init__(
        self,
        in_dim,
        bottleneck_dim=128
    ):
        super().__init__()

        self.linear1 = nn.Conv1d(
            in_dim,
            bottleneck_dim,
            kernel_size=1
        )

        self.linear2 = nn.Conv1d(
            bottleneck_dim,
            in_dim,
            kernel_size=1
        )

    def forward(self, x):

        alpha = torch.tanh(
            self.linear1(x)
        )

        alpha = torch.softmax(
            self.linear2(alpha),
            dim=2
        )

        mean = torch.sum(
            alpha * x,
            dim=2
        )

        var = (
            torch.sum(
                alpha * (x**2),
                dim=2
            )
            - mean**2
        )

        std = torch.sqrt(
            var.clamp(min=1e-7)
        )

        return torch.cat(
            [mean, std],
            dim=1
        )

In [4]:
class AAMSoftmax(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_classes,
        margin=0.2,
        scale=30
    ):
        super().__init__()

        self.margin = margin
        self.scale = scale

        self.weight = nn.Parameter(
            torch.randn(
                num_classes,
                embedding_dim
            )
        )

        nn.init.xavier_normal_(
            self.weight
        )

    def forward(
        self,
        embeddings,
        labels
    ):

        embeddings = F.normalize(
            embeddings,
            dim=1
        )

        weight = F.normalize(
            self.weight,
            dim=1
        )

        cosine = F.linear(
            embeddings,
            weight
        )

        theta = torch.acos(
            torch.clamp(
                cosine,
                -1 + 1e-7,
                1 - 1e-7
            )
        )

        target_cosine = torch.cos(
            theta + self.margin
        )

        one_hot = torch.zeros_like(
            cosine
        )

        one_hot.scatter_(
            1,
            labels.view(-1,1),
            1
        )

        logits = (
            one_hot * target_cosine
            +
            (1-one_hot) * cosine
        )

        logits *= self.scale

        return logits

In [5]:
class WavLMSpeakerModel(
    nn.Module
):

    def __init__(
        self,
        num_speakers
    ):
        super().__init__()

        self.wavlm = (
            WavLMModel
            .from_pretrained(
                "microsoft/wavlm-base"
            )
        )

        self.layer_weights = (
            nn.Parameter(
                torch.ones(13)
            )
        )

        self.pooling = ASTP(768)

        self.aam = AAMSoftmax(
            embedding_dim=1536,
            num_classes=num_speakers
        )

    def forward(
        self,
        input_values,
        labels=None
    ):

        outputs = self.wavlm(
            input_values=input_values,
            output_hidden_states=True
        )

        hidden_states = (
            outputs.hidden_states
        )

        weights = torch.softmax(
            self.layer_weights,
            dim=0
        )

        combined = 0

        for w, h in zip(
            weights,
            hidden_states
        ):
            combined += w*h

        combined = (
            combined
            .transpose(1,2)
        )

        embedding = self.pooling(
            combined
        )

        embedding = F.normalize(
            embedding,
            dim=-1
        )

        logits = None

        if labels is not None:

            logits = self.aam(
                embedding,
                labels
            )

        return logits, embedding

In [6]:
processor = (
    Wav2Vec2FeatureExtractor
    .from_pretrained(
        "microsoft/wavlm-base"
    )
)

def collate_fn(batch):

    audios = [x[0] for x in batch]

    labels = torch.tensor(
        [x[1] for x in batch]
    )

    features = processor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    return (
        features.input_values,
        labels
    )

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [7]:
class VoxCelebDataset(Dataset):

    def __init__(self):

        self.samples = []

        for speaker in sorted(
            os.listdir(VOX_PATH)
        ):

            if speaker not in spk2idx:
                continue

            speaker_path = os.path.join(
                VOX_PATH,
                speaker
            )

            for video in os.listdir(
                speaker_path
            ):

                video_path = os.path.join(
                    speaker_path,
                    video
                )

                for wav in os.listdir(
                    video_path
                ):

                    self.samples.append(
                        (
                            os.path.join(
                                video_path,
                                wav
                            ),
                            spk2idx[speaker]
                        )
                    )

    def __len__(self):

        return len(
            self.samples
        )

    def __getitem__(self, idx):

        wav_path, label = (
            self.samples[idx]
        )

        audio, sr = sf.read(
            wav_path
        )

        if len(audio.shape) > 1:
            audio = audio.mean(
                axis=1
            )

        audio = audio.astype(
            np.float32
        )

        target_len = 48000

        if len(audio) > target_len:

            start = np.random.randint(
                0,
                len(audio) - target_len
            )

            audio = audio[
                start:start+target_len
            ]

        else:

            audio = np.pad(
                audio,
                (
                    0,
                    target_len-len(audio)
                )
            )

        return audio, label

In [8]:
model = WavLMSpeakerModel(
    num_speakers=num_speakers
)

model.load_state_dict(
    checkpoint["model"]
)

print(
    torch.softmax(
        model.layer_weights,
        dim=0
    )
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

tensor([0.0689, 0.2940, 0.1046, 0.0679, 0.0594, 0.0556, 0.0517, 0.0514, 0.0502,
        0.0484, 0.0497, 0.0497, 0.0485], grad_fn=<SoftmaxBackward0>)


In [9]:
dataset = VoxCelebDataset()

print(
    "Training samples:",
    len(dataset)
)

print(
    "Speakers:",
    num_speakers
)

print(
    "First sample:"
)

print(
    dataset.samples[0]
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

sample_x, sample_y = next(
    iter(loader)
)

print(sample_x.shape)
print(sample_y.shape)

Training samples: 23749
Speakers: 1211
First sample:
('/kaggle/input/datasets/arie07/audiodataset10percent/VoxCeleb/vox1_dev_wav/id10041/0-KA7pTDABQ/00002.wav', 0)
torch.Size([8, 48000])
torch.Size([8])


In [10]:
model = WavLMSpeakerModel(
    num_speakers=num_speakers
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

print("Loading local baseline...")

checkpoint = torch.load(
    BASELINE_CHECKPOINT,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model"]
)

print("Baseline loaded.")

#################################################
# Freeze all WavLM
#################################################

for p in model.wavlm.parameters():
    p.requires_grad = False

#################################################
# Unfreeze last 3 layers
#################################################

for p in model.wavlm.encoder.layers[-3:].parameters():
    p.requires_grad = True

#################################################
# Train Layer Weights
#################################################

model.layer_weights.requires_grad = True

#################################################
# Train ASTP
#################################################

for p in model.pooling.parameters():
    p.requires_grad = True

#################################################
# Train AAM
#################################################

for p in model.aam.parameters():
    p.requires_grad = True

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Trainable: {trainable:,}"
)

print(
    f"Percent: {100*trainable/total:.2f}%"
)

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-5
)

step = 0

if os.path.exists(
    PARTIAL_FT_CHECKPOINT
):

    print(
        "Resuming Partial FT..."
    )

    checkpoint = torch.load(
        PARTIAL_FT_CHECKPOINT,
        map_location=DEVICE
    )

    model.load_state_dict(
        checkpoint["model"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    step = checkpoint["step"]

    print(
        f"Resuming from step {step}"
    )

num_steps_to_train = 20000

running_loss = 0.0

model.train()

stop_training = False

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

Loading local baseline...
Baseline loaded.
Trainable: 23,322,825
Percent: 24.18%


In [11]:
print(
    torch.softmax(
        model.layer_weights,
        dim=0
    ).detach().cpu()
)

tensor([0.0689, 0.2940, 0.1046, 0.0679, 0.0594, 0.0556, 0.0517, 0.0514, 0.0502,
        0.0484, 0.0497, 0.0497, 0.0485])


In [12]:
for epoch in range(100):

    if stop_training:
        break

    print(
        f"\n--- Epoch {epoch+1} ---"
    )

    for input_values, labels in loader:

        input_values = (
            input_values.to(
                DEVICE
            )
        )

        labels = labels.to(
            DEVICE
        )

        logits, embeddings = model(
            input_values,
            labels
        )

        loss = criterion(
            logits,
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += (
            loss.item()
        )

        step += 1

        if step % 100 == 0:

            avg_loss = (
                running_loss / 100
            )

            print(
                f"Step {step} | "
                f"Avg Loss = "
                f"{avg_loss:.4f}"
            )

            running_loss = 0.0

        if step % 500 == 0:

            torch.save(
        {
            "model":
                model.state_dict(),

            "optimizer":
                optimizer.state_dict(),

            "step":
                step
        },
        PARTIAL_FT_CHECKPOINT
    )

            print(
        f"Checkpoint saved "
        f"at step {step}"
    )

        if step >= num_steps_to_train:

            stop_training = True
            break

torch.save(
    {
        "model":
            model.state_dict(),

        "optimizer":
            optimizer.state_dict(),

        "step":
            step
    },
    PARTIAL_FT_CHECKPOINT
)

print(
    "\nTraining completed!"
)

print(
    "Final layer weights:"
)

print(
    torch.softmax(
        model.layer_weights,
        dim=0
    ).detach().cpu()
)


--- Epoch 1 ---
Step 100 | Avg Loss = 8.9888
Step 200 | Avg Loss = 8.9343
Step 300 | Avg Loss = 8.9001
Step 400 | Avg Loss = 8.8414
Step 500 | Avg Loss = 8.8965
Checkpoint saved at step 500
Step 600 | Avg Loss = 8.8376
Step 700 | Avg Loss = 8.8739
Step 800 | Avg Loss = 8.9721
Step 900 | Avg Loss = 8.9607
Step 1000 | Avg Loss = 8.8890
Checkpoint saved at step 1000
Step 1100 | Avg Loss = 8.8977
Step 1200 | Avg Loss = 8.7341
Step 1300 | Avg Loss = 8.7834
Step 1400 | Avg Loss = 8.7552
Step 1500 | Avg Loss = 8.9637
Checkpoint saved at step 1500
Step 1600 | Avg Loss = 8.8044
Step 1700 | Avg Loss = 8.8683
Step 1800 | Avg Loss = 8.8009
Step 1900 | Avg Loss = 8.8499
Step 2000 | Avg Loss = 8.7638
Checkpoint saved at step 2000
Step 2100 | Avg Loss = 8.7210
Step 2200 | Avg Loss = 8.8487
Step 2300 | Avg Loss = 8.6281
Step 2400 | Avg Loss = 8.8696
Step 2500 | Avg Loss = 8.7271
Checkpoint saved at step 2500
Step 2600 | Avg Loss = 8.7208
Step 2700 | Avg Loss = 8.6888
Step 2800 | Avg Loss = 8.8684
Ste